# 02 — Feature Engineering

Cleaning, time-feature extraction, rolling statistics, and stationarity testing for ARIMA readiness.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('../data/processed/ad_metrics_clean.csv', parse_dates=['datetime'])
metrics = ['ecpm', 'fill_rate', 'ctr', 'impressions', 'arpdau']
print(f'Shape: {df.shape}')
df[metrics].describe().round(4)

## Stationarity Testing (Augmented Dickey-Fuller)

ARIMA requires (or benefits from) stationary time series. We check each metric.

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    p = result[1]
    stat = 'Stationary ✓' if p < 0.05 else 'Non-Stationary ✗'
    print(f'{name:15s} | ADF Stat: {result[0]:8.4f} | p-value: {p:.4f} | {stat}')

print('Augmented Dickey-Fuller Test Results')
print('-' * 65)
for m in metrics:
    adf_test(df[m], m.upper())

## ACF / PACF — Choosing ARIMA (p, d, q)

In [ ]:
# Show ACF/PACF for eCPM (primary revenue metric)
series = df['ecpm'].diff().dropna()   # first-differenced for stationarity

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(series, lags=48, ax=ax1, alpha=0.05)
ax1.set_title('ACF — eCPM (differenced)', fontweight='bold')

plot_pacf(series, lags=48, ax=ax2, alpha=0.05, method='ywm')
ax2.set_title('PACF — eCPM (differenced)', fontweight='bold')

plt.tight_layout()
plt.show()
print('Reading: ACF cuts off at lag ~2 → MA(2); PACF cuts off at lag ~2 → AR(2) → ARIMA(2,1,2)')

## Rolling Statistics Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(df['datetime'], df['ecpm'], linewidth=0.8, label='Observed eCPM', color='#555')
axes[0].plot(df['datetime'], df['ecpm_roll_mean'], linewidth=1.2, label='24h Rolling Mean', color='#2196F3')
axes[0].fill_between(
    df['datetime'],
    df['ecpm_roll_mean'] - 2 * df['ecpm_roll_std'],
    df['ecpm_roll_mean'] + 2 * df['ecpm_roll_std'],
    alpha=0.15, color='#2196F3', label='±2σ Band'
)
axes[0].set_title('eCPM with Rolling Mean ± 2σ Band', fontweight='bold')
axes[0].set_ylabel('eCPM ($)')
axes[0].legend()

axes[1].plot(df['datetime'], df['fill_rate'], linewidth=0.8, label='Observed Fill Rate', color='#555')
axes[1].plot(df['datetime'], df['fill_rate_roll_mean'], linewidth=1.2, label='24h Rolling Mean', color='#e07b54')
axes[1].fill_between(
    df['datetime'],
    df['fill_rate_roll_mean'] - 2 * df['fill_rate_roll_std'],
    df['fill_rate_roll_mean'] + 2 * df['fill_rate_roll_std'],
    alpha=0.15, color='#e07b54', label='±2σ Band'
)
axes[1].set_title('Fill Rate with Rolling Mean ± 2σ Band', fontweight='bold')
axes[1].set_ylabel('Fill Rate')
axes[1].legend()

plt.xlabel('Datetime')
plt.tight_layout()
plt.show()